### Mildew Prediction & Live Inference Pipeline

#### Objectives
* Configure modern TensorFlow 2.16+ / Keras 3 environment and Windows DLL handling.
* Load the trained model (`powdery_mildew_detector_model.h5`) and class index mapping (`class_indices.pkl`) from `outputs/v1/`.
* Define an image preprocessing function utilizing `keras.utils` (`load_img`, `img_to_array`) aligned with the model's built-in `Rescaling` layer.
* Construct modular prediction and visualization helper functions to execute single and batch leaf evaluations.
* Validate inference performance on unseen test images before refactoring logic into `src/machine_learning.py` for Streamlit deployment.

#### Inputs
* Trained model artifact: `outputs/v1/powdery_mildew_detector_model.h5`
* Class mappings: `outputs/v1/class_indices.pkl`
* Test images from `inputs/cherry_leaves/test/` or live user uploads

#### Outputs
* Structured prediction DataFrames and visual diagnostic plots with confidence scores

In [1]:
import os
import sys
import random
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# 1. Resolve Windows DLL Loading and Mute Logs
if sys.platform == "win32":
    os.add_dll_directory(r"C:\Windows\System32")

os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
import keras
from keras.models import load_model
from keras.utils import load_img, img_to_array

# 2. Resolve Project Root Directory
CURRENT_DIR = os.getcwd()
PROJECT_DIR = os.path.dirname(CURRENT_DIR) if os.path.basename(CURRENT_DIR) == 'jupyter_notebooks' else CURRENT_DIR

# 3. Set Paths
OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs', 'v1')
MODEL_PATH = os.path.join(OUTPUTS_DIR, 'powdery_mildew_detector_model.h5')
CLASS_INDICES_PATH = os.path.join(OUTPUTS_DIR, 'class_indices.pkl')
TEST_DIR = os.path.join(PROJECT_DIR, 'inputs', 'cherry_leaves', 'test')

# 4. Load Trained Model & Class Mapping
model = load_model(MODEL_PATH)

with open(CLASS_INDICES_PATH, 'rb') as f:
    class_indices = pickle.load(f)

# Invert mapping: {0: 'healthy', 1: 'powdery_mildew'}
map_labels = {v: k for k, v in class_indices.items()}

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")
print("Model and class indices loaded successfully.")
print(f"Class Map: {map_labels}")

TensorFlow Version: 2.19.1
Keras Version: 3.15.1
Model and class indices loaded successfully.
Class Map: {0: 'healthy', 1: 'powdery_mildew'}


In [2]:
def load_and_preprocess_image(img_path, target_size=(256, 256)):
    """
    Loads an image and converts it into a 4D tensor [1, 256, 256, 3].
    Note: Pixel division by 255 is omitted here because standard array values [0, 255]
    are rescaled automatically by the model's internal Rescaling layer.
    """
    # Load PIL image for visualization
    img_pil = load_img(img_path, target_size=target_size)
    
    # Convert PIL Image to Numpy Array [256, 256, 3] with values in [0, 255]
    img_array = img_to_array(img_pil)
    
    # Expand dimensions to batch tensor [1, 256, 256, 3]
    img_tensor = np.expand_dims(img_array, axis=0)
    
    return img_pil, img_tensor